# 1. StrOutputParser

**The single most-used parser in all of LangChain.** It does one humble but essential job: take the
chat model's message object and give you back the **plain string** inside it.

---

## 1. Simple Definition

> **Kid version:** When the AI answers, it doesn't just hand you words — it hands you the words inside
> a **box** with stickers all over it (who sent it, token counts, etc.). Most of the time you just
> want the words. `StrOutputParser` opens the box and gives you **only the words**.

**Professional definition:** `StrOutputParser` extracts the text `content` from a model's output (an
`AIMessage` from a chat model, or a raw string from an LLM) and returns it as a plain `str`.

```python
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

chain = ChatPromptTemplate.from_template("Say hi to {name}") | ChatOpenAI() | StrOutputParser()
print(chain.invoke({"name": "Sam"}))   # 'Hi Sam!'  ← a str, not an AIMessage
```

---

## 2. Why Does It Exist?

**The problem:** Chat models return an `AIMessage` object, not a string. If you feed a chain's result
into another prompt, into `print`, or into any code expecting text, the message object gets in the
way (`.content`, metadata, etc.).

### Before

```python
resp = (prompt | model).invoke({...})   # AIMessage(content='Hi Sam!', response_metadata=...)
text = resp.content                       # you manually pull .content everywhere
```

### After

```python
text = (prompt | model | StrOutputParser()).invoke({...})   # 'Hi Sam!' directly
```

It removes the constant `.content` boilerplate and makes chains compose cleanly (the next step gets a
clean string). It's the default final step for any **text-generation** chain.

---

## 3. Real-Life Analogy

**Unwrapping a delivery** 📦. The parcel arrives wrapped in a box with labels, tracking stickers, and
padding (the `AIMessage` metadata). You only care about the item inside (the text). `StrOutputParser`
unwraps it and hands you the item.

---

## 4. Where It Fits in LangChain Architecture

```
BaseOutputParser
    │
    ▼
BaseTransformOutputParser        ← supports streaming
    │
    ▼
StrOutputParser                  → str
```

- Inherits streaming support from `BaseTransformOutputParser`, so it **passes tokens straight through**
  when you `.stream()` a chain — great for typing-effect UIs.
- It's the **odd one out**: it has no meaningful `get_format_instructions()` (any text is valid) — its
  whole job is Job 2 (extract text).

---

## 5. Internal Working

```
  model output
     AIMessage(content="Hi Sam!", response_metadata={...})
        │
        ▼
  StrOutputParser
     → read .content
     → (if already a plain string, return as-is)
        │
        ▼
  "Hi Sam!"   (str)

  STREAMING:
     token "Hi" → yield "Hi"
     token " Sam!" → yield " Sam!"     (passes each chunk through unchanged)
```

---

## 6. Attributes / Methods

`StrOutputParser` takes **no configuration** — that's the point. Its behavior comes from the base
interface.

### `parse()`

**Definition:** Returns the input text unchanged (as a string).

**Why it exists:** Completes the `BaseOutputParser` contract.

```python
StrOutputParser().parse("hello")   # 'hello'
```

In [2]:
from langchain_core.output_parsers import StrOutputParser
StrOutputParser().parse("Hello, World!")

'Hello, World!'

### `invoke() / streaming`

**Definition:** As a Runnable, it extracts `.content` on `.invoke()` and passes chunks through on
`.stream()`.

**Why it exists:** Lets it sit at the end of a chain and support live output.

**Real-life use case:** A chatbot UI that prints the answer token-by-token.

```python
for chunk in chain.stream({"name": "Sam"}):
    print(chunk, end="", flush=True)   # streams the string as it generates
```

In [3]:
# let's see the output of this simple code without any output parser. so it is returning a AI model output object.

from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate


# Initialize the language model
llm = ChatOllama(model="qwen3:8b")

# prompt
prompt = PromptTemplate(template="Explain Python in one sentence.")

chain = prompt | llm

llm_result = chain.invoke({})

llm_result

AIMessage(content='Python is a high-level, interpreted programming language known for its readability, simplicity, and versatility in applications like web development, data analysis, artificial intelligence, and automation.', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-09-04T09:34:06.9600235Z', 'done': True, 'done_reason': 'stop', 'total_duration': 30570457100, 'load_duration': 5088924300, 'prompt_eval_count': 17, 'prompt_eval_duration': 323049000, 'eval_count': 329, 'eval_duration': 25153582000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--01a06bc4-3704-7a71-ab48-f8c79bd7219c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 329, 'total_tokens': 346})

In [4]:
# let's see the output of this simple code with output parser. so it is returning a string output instead of AI model output object.

from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize the language model
llm = ChatOllama(model="qwen3:8b")

# prompt
prompt = PromptTemplate(template="Explain Python in one sentence.")

parser = StrOutputParser()

chain = prompt | llm | parser

llm_result = chain.invoke({})

llm_result

'Python is a high-level, interpreted programming language known for its readability, simplicity, and versatility, widely used in web development, data analysis, artificial intelligence, and automation.'

In [5]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize the language model
llm = ChatOllama(model="qwen3:8b")

# 1st prompt -> detailed report
detailed_report_prompt = PromptTemplate(template='Write a detailed report on {topic}', input_variables=['topic'])

# 2nd prompt -> summary
summary_prompt = PromptTemplate(template='Write a 5 line summary on the following text. \n\n {detailed_report}', input_variables=['detailed_report'])

parser = StrOutputParser()

chain = detailed_report_prompt | llm | parser | summary_prompt | llm | parser

llm_result = chain.invoke({'topic':'black hole'})

llm_result

'Black holes, predicted by Einstein’s theory and confirmed by observations, are regions of spacetime with gravity so intense that not even light can escape. They form from stellar collapse or cosmic processes, existing in stellar-mass, supermassive, and intermediate categories. Observed through gravitational waves, accretion disks, and jets, their structure includes event horizons, singularities, and theoretical complexities like Hawking radiation and the information paradox. Research continues to unravel their role in galaxy evolution, cosmic structure, and the unification of general relativity with quantum mechanics.'